### Code Hist.

 - CODE  
    &ensp; : Model - KIER Method 02(Clustering)

  - DATE      &ensp; 2023-03-05 Created  
    &emsp;&emsp;&emsp;&emsp;&emsp;&emsp; 1) Dataset : KIER / Seq2Seq    
    &emsp;&emsp;&emsp;&emsp;&emsp;&emsp; 2) Model : LightGBM  
    &emsp;&emsp;&emsp;&emsp;&emsp;&emsp; 3)   

 - Related Link  
    &ensp; : 

# 01. Code

## 01-01. Init

### 01-01-01. Init_Module Import

In [ ]:
#region Basic_Import
## Basic
import os, sys, warnings
os.environ["CUDA_VISIBLE_DEVICES"] = "0"
os.path.dirname(os.path.abspath('./__file__'))
sys.path.append(os.path.dirname(os.path.abspath(os.path.dirname('./__file__'))))
warnings.filterwarnings('ignore')

import numpy as np, pandas as pd
from pandas import DataFrame, Series
pd.options.display.float_format = '{:.10f}'.format

import math, random

## Datetime
import time, datetime as dt
from datetime import datetime, date, timedelta

## glob
import glob, requests, json
from glob import glob

## 시각화
import matplotlib.pyplot as plt, seaborn as sns
# %matplotlib inline
plt.rcParams['figure.figsize'] = [10, 8]

from scipy import stats

## Split, 정규화
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import MinMaxScaler, StandardScaler

# K-Means 알고리즘
from sklearn.cluster import KMeans, MiniBatchKMeans

# Clustering 알고리즘의 성능 평가 측도
from sklearn import metrics
from sklearn.metrics import homogeneity_score, completeness_score, v_measure_score, adjusted_rand_score, silhouette_score, rand_score, calinski_harabasz_score, davies_bouldin_score
from sklearn.metrics.cluster import contingency_matrix

## For Web
import urllib
from urllib.request import urlopen
from urllib.parse import urlencode, unquote, quote_plus
from selenium import webdriver
from selenium.webdriver.chrome.service import Service
from webdriver_manager.chrome import ChromeDriverManager
from bs4 import BeautifulSoup

import tqdm
from tqdm.notebook import tqdm
#endregion Basic_Import

In [ ]:
## Import_DL
str_tar = "tf"
## For Torch
if str_tar == "torch":
    import torch, torch.nn as nn
    from torch.nn.utils import weight_norm
    print("Torch Imported")
## For TF
elif str_tar == "tf":
    import tensorflow as tf, tensorflow_addons as tfa
    from keras.callbacks import EarlyStopping, ModelCheckpoint
    from keras.models import Sequential, load_model
    from keras_flops import get_flops
    print("Tensorflow Imported")
else:
    print("Error : Cannot be used except for Keywords")
    print(" : torch / tf")

In [ ]:
## LSTM
from tensorflow import keras
from tensorflow.keras import Sequential, layers, callbacks
from tensorflow.keras.layers import Dense, LSTM, Dropout, GRU, Bidirectional
from keras.callbacks import EarlyStopping, ModelCheckpoint

In [ ]:
## Import_Local
from core import data_datetime as com_date, provider_kma as com_KMA, provider_keco as com_KECO, provider_kasi as com_Holi, provider_kier_m02 as com_KIER_M02, data_clustering as com_clustering, data_analysis as com_Analysis, model_ml as com_Model

### 01-01-02. Config (Directory, Params)

In [ ]:
## Init_config
SEED = 42

np.random.seed(SEED)
tf.random.set_seed(SEED)
random.seed(SEED)
os.environ["PYTHONHASHSEED"], os.environ['TF_DETERMINISTIC_OPS'] = str(SEED), "1"

In [ ]:
## Define Todate str
str_now_ymd = pd.datetime.now().date()
str_now_y, str_now_m, str_now_d = pd.datetime.now().year, pd.datetime.now().month, pd.datetime.now().day
str_now_hr, str_now_min = pd.datetime.now().hour, pd.datetime.now().minute

print(pd.datetime.now())
print(str(str_now_y) + " / " + str(str_now_m)  + " / " + str(str_now_d))
print(str(str_now_hr) + " : " + str(str_now_min))

In [ ]:
## Dict_Domain
## {0:"ELEC", 1:"HEAT", 2:"WATER", 3:"HOT_HEAT", 4:"HOT_FLOW", 99:"GAS"}
## {0 : '10MIN', 1 : '30MIN', 2 : '1H', 3 : '12H', 4 : '1D', 5 : '1W', 6 : '2W', 7 : '1M'}
int_domain, int_interval, K = 0, 0, 2

## Domain, ACCU/INST Column
str_domain, str_col_accu, str_col_inst = com_KIER_M02.create_domain_str(int_domain)
## Directory Root
str_dirData, str_dir_raw, str_dir_cleansed, str_dirName_bld, str_dirName_h = com_KIER_M02.create_dir_str(str_domain)
## Interval, Target File
str_interval, str_fileRaw, str_fileRaw_hList, str_file = com_KIER_M02.create_file_str(str_domain, int_interval)

print(str(os.listdir(str_dirData)) + "\n")
print(os.listdir(str_dirName_h))

## 01-02. Data Load (df_raw)

### 01-02-01. KMA ASOS

In [ ]:
## KMA_ASOS Data
# str_dir_kmaAsos = "../data/data_KMA_ASOS/"

## Interpolate / Filled ASOS Data
str_file = '../data_Energy_KIER/KMA_ASOS_119_2010_2023_1st_to CSV.csv'
df_ASOS = pd.read_csv(str_file, index_col = 0).reset_index()

try : df_ASOS['METER_DATE'] = pd.to_datetime(df_ASOS['METER_DATE'])
except KeyError : df_kier_raw = com_date.create_col_datetime(df_ASOS, 'METER_DATE', 'YEAR', 'MONTH', 'DAY', 'HOUR', 'MINUTE').drop(labels = ['None'], axis = 1)

df_ASOS

In [ ]:
## Cluster 기준 Interval
str_file_clustering = 'KIER_' + str(str_domain) + '_Labeled_' + str_interval + '_K' + str(K) + '.csv'
df_kier_h_cluster = pd.read_csv(str_dirName_h + str_file_clustering
                                , index_col = 0).rename(columns = {'index' : 'h_index'})[['h_index', 'target_' + str_domain]]
print(str_interval)
print(df_kier_h_cluster['target_' + str_domain].drop_duplicates())
df_kier_h_cluster

In [ ]:
list_kier_h_all = df_kier_h_cluster['h_index']
print(len(list_kier_h_all))
list_kier_h_c0 = df_kier_h_cluster[df_kier_h_cluster['target_' + str_domain] == 0]['h_index']
print(len(list_kier_h_c0))
list_kier_h_c1 = df_kier_h_cluster[df_kier_h_cluster['target_' + str_domain] == 1]['h_index']
print(len(list_kier_h_c1))
list_kier_h_c2 = df_kier_h_cluster[df_kier_h_cluster['target_' + str_domain] == 2]['h_index']
print(len(list_kier_h_c2))

In [ ]:
## 사용량 Data Load
## 1시간 단위
str_file = 'KIER_' + str_domain + '_INST_1H_Resampled.csv'
df_raw = pd.read_csv(str_dirName_h + str_file, index_col = 0)
df_raw

In [ ]:
## 전체 사용량 합계
df_kier_h_all = df_raw.copy()
df_kier_h_all['METER_DATE'] = pd.to_datetime(df_kier_h_all['METER_DATE'])
df_kier_h_tmp = df_raw[list_kier_h_all]
df_kier_h_all[str_domain + '_INST_SUM_ALL'] = df_kier_h_tmp.sum(axis = 1)
## 시점을 밀어서, 세대별 사용량을 과거 사용량으로 사용
df_kier_h_all[str_domain + '_INST_SUM_ALL'] = df_kier_h_all[str_domain + '_INST_SUM_ALL'].shift(1)
df_kier_h_all.dropna()
df_kier_h_all = df_kier_h_all.dropna()

In [ ]:
## Cluster별 사용량 합계
## ■ C00
df_kier_h_c0 = df_raw.copy()[list_kier_h_c0]
df_kier_h_c0['METER_DATE'] = pd.to_datetime(df_kier_h_all['METER_DATE'])
df_kier_h_tmp = df_raw[list_kier_h_c0]
df_kier_h_c0[str_domain + '_INST_SUM_C0'] = df_kier_h_tmp.sum(axis = 1)
## 시점을 밀어서, 세대별 사용량을 과거 사용량으로 사용
df_kier_h_c0[str_domain + '_INST_SUM_C0'] = df_kier_h_c0[str_domain + '_INST_SUM_C0'].shift(1)
df_kier_h_c0 = df_kier_h_c0.dropna()

## ■ C01
df_kier_h_c1 = df_raw.copy()[list_kier_h_c1]
df_kier_h_c1['METER_DATE'] = pd.to_datetime(df_kier_h_all['METER_DATE'])
df_kier_h_tmp = df_raw[list_kier_h_c1]
df_kier_h_c1[str_domain + '_INST_SUM_C1'] = df_kier_h_tmp.sum(axis = 1)
## 시점을 밀어서, 세대별 사용량을 과거 사용량으로 사용
df_kier_h_c1[str_domain + '_INST_SUM_C1'] = df_kier_h_c1[str_domain + '_INST_SUM_C1'].shift(1)
df_kier_h_c1 = df_kier_h_c1.dropna()

if K == 3 : 
    ## ■ C02
    df_kier_h_c2 = df_raw.copy()[list_kier_h_c2]
    df_kier_h_c2['METER_DATE'] = pd.to_datetime(df_kier_h_all['METER_DATE'])
    df_kier_h_tmp = df_raw[list_kier_h_c2]
    df_kier_h_c2[str_domain + '_INST_SUM_C2'] = df_kier_h_tmp.sum(axis = 1)
    ## 시점을 밀어서, 세대별 사용량을 과거 사용량으로 사용
    df_kier_h_c2[str_domain + '_INST_SUM_C2'] = df_kier_h_c2[str_domain + '_INST_SUM_C2'].shift(1)
    df_kier_h_c2 = df_kier_h_c2.dropna()

In [ ]:
df_kier_h_c0

In [ ]:
## 날씨 데이터 추가
df_kier_h_all = pd.merge(df_kier_h_all, df_ASOS, how = 'left', on = ['METER_DATE'])
df_kier_h_all = com_KMA.Interpolate_KMA_ASOS(df_kier_h_all)
df_kier_h_all = com_date.create_col_ymdhm(df_kier_h_all, 'METER_DATE')

df_kier_h_c0 = pd.merge(df_kier_h_c0, df_ASOS, how = 'left', on = ['METER_DATE'])
df_kier_h_c0 = com_KMA.Interpolate_KMA_ASOS(df_kier_h_c0)
df_kier_h_c0 = com_date.create_col_ymdhm(df_kier_h_c0, 'METER_DATE')

df_kier_h_c1 = pd.merge(df_kier_h_c1, df_ASOS, how = 'left', on = ['METER_DATE'])
df_kier_h_c1 = com_KMA.Interpolate_KMA_ASOS(df_kier_h_c1)
df_kier_h_c1 = com_date.create_col_ymdhm(df_kier_h_c1, 'METER_DATE')

if K == 3 : 
    df_kier_h_c2 = pd.merge(df_kier_h_c2, df_ASOS, how = 'left', on = ['METER_DATE'])
    df_kier_h_c2 = com_KMA.Interpolate_KMA_ASOS(df_kier_h_c2)
    df_kier_h_c2 = com_date.create_col_ymdhm(df_kier_h_c2, 'METER_DATE')

In [ ]:
## 각 세대별 이전시간 사용량 컬럼을 제거하고 분석하는 경우
df_kier_h_all = df_kier_h_all.drop(labels = list_kier_h_all, axis = 1)
df_kier_h_c0 = df_kier_h_c0.drop(labels = list_kier_h_c0, axis = 1)
df_kier_h_c1 = df_kier_h_c1.drop(labels = list_kier_h_c1, axis = 1)
if K == 3 : df_kier_h_c2 = df_kier_h_c2.drop(labels = list_kier_h_c2, axis = 1)

In [ ]:
# pd.set_option('display.max_row', 500)
print(df_kier_h_all.shape, ' /// ', df_kier_h_all.columns)
print(df_kier_h_all.isna().sum())
print(df_kier_h_c0.shape, ' /// ', df_kier_h_all.columns)
print(df_kier_h_c0.isna().sum())
print(df_kier_h_c1.shape, ' /// ', df_kier_h_all.columns)
print(df_kier_h_c1.isna().sum())

if K == 3 : 
    print(df_kier_h_c2.shape, ' /// ', df_kier_h_all.columns)
    print(df_kier_h_c2.isna().sum())

## 01-06. Data Split (Train/Test Setting)

In [ ]:
## 모든 세대
df_raw = df_kier_h_all
str_col_tar = str_domain + '_INST_SUM_ALL'
## C1 세대
# df_raw = df_kier_h_c0
# str_col_tar = str_domain + '_INST_SUM_C0'
## C2 세대
# df_raw = df_kier_h_c1
# str_col_tar = str_domain + '_INST_SUM_C1'
## C3 세대
# df_raw = df_kier_h_c2
# str_col_tar = str_domain + '_INST_SUM_C2'

# df_raw = com_date.create_col_ymdhm(df_raw, 'METER_DATE')
# df_raw = com_date.create_col_weekdays(df_raw, 'METER_DATE')
df_raw = df_raw.drop(columns = ['METER_DATE', 'DAY']).dropna()
# df_raw = df_raw.drop(columns = ['METER_DATE', 'day_of_the_week'])
# df_raw = df_raw.drop(columns = ['METER_DATE', 'YEAR', 'MONTH', 'DAY', 'HOUR', 'None']).dropna()

trainSet_Origin, testSet_Origin = train_test_split(df_raw, test_size=0.3, shuffle=False)
print(trainSet_Origin.shape, testSet_Origin.shape)

In [ ]:
trainSet = trainSet_Origin
testSet = testSet_Origin

## Input / Target Split
trainXX = trainSet.drop([str_col_tar],axis=1)
trainYY = trainSet[[str_col_tar]]
testXX = testSet.drop([str_col_tar],axis=1)
testYY = testSet[[str_col_tar]]

In [ ]:
trainXXindex = trainXX.index
trainXXcolumns = trainXX.columns

trainYYindex = trainYY.index
trainYYcolumns = trainYY.columns

testXXindex = testXX.index
testXXcolumns = testXX.columns

testYYindex = testYY.index
testYYcolumns = testYY.columns

In [ ]:
d_trainXX=pd.DataFrame(trainXX, index=trainXXindex, columns=trainXXcolumns)
d_trainYY=trainYY

d_testXX=pd.DataFrame(testXX, index=testXXindex, columns=testXXcolumns)
d_testYY=testYY

In [ ]:
int_len_col_input = len(trainXXcolumns)
int_len_col_input

In [ ]:
def buildDataSet(input, target, seqLength):
    xdata = []
    ydata = []
    for i in range(len(input) - seqLength):
        tx = input.iloc[i:i+seqLength]
        ty = target.iloc[i+seqLength-1]
        xdata.append(tx)
        ydata.append(ty)
    return np.array(xdata), np.array(ydata)

In [ ]:
trainX_kier, trainY_kier = buildDataSet(d_trainXX, d_trainYY, 24)
testX_kier, testY_kier = buildDataSet(d_testXX, d_testYY, 24)

## 01-07. DL (Seq2Seq)


In [ ]:
import time
tm_start = time.time()

In [ ]:
# def seq2seq_model(input_shape):
#     model_input = tf.keras.layers.Input(shape=input_shape)

#     # for feature extracting
#     conv1 = tf.keras.layers.Conv1D(128, 1, activation='swish')(model_input)
#     pool1 = tf.keras.layers.MaxPool1D(pool_size=2, strides=1, padding='same')(conv1)
#     conv2 = tf.keras.layers.Conv1D(64, 1, activation='swish')(pool1)
#     pool2 = tf.keras.layers.MaxPool1D(pool_size=2, strides=1, padding='same')(conv2)
#     conv3 = tf.keras.layers.Conv1D(32, 1, activation='swish')(pool2)
#     pool3 = tf.keras.layers.MaxPool1D(pool_size=2, strides=1, padding='same')(conv3)

#     # 인코더 - 디코더 선언
#     encoder_lstm1 = tf.keras.layers.LSTM(32, return_sequences=True, activation='tanh')
#     encoder_lstm2 = tf.keras.layers.LSTM(64, return_sequences=True, activation='tanh')
#     encoder_lstm3 = tf.keras.layers.LSTM(128, return_state=True, return_sequences=True, activation='tanh')

#     decoder_lstm1 = tf.keras.layers.LSTM(128, return_sequences=True, activation='tanh')
#     decoder_lstm2 = tf.keras.layers.LSTM(64, return_sequences=True, activation='tanh')
#     decoder_lstm3 = tf.keras.layers.LSTM(32, return_sequences=True, activation='tanh')

#     # 인코더
#     encoder_output_lstm1 = encoder_lstm1(pool3)
#     encoder_output_lstm2 = encoder_lstm2(pool2)
#     encoder_output_lstm3, state_h, state_c = encoder_lstm3(encoder_output_lstm2)

#     #디코더
#     decoder_lstm1_output = decoder_lstm1(encoder_output_lstm3, initial_state=[state_h, state_c])
#     decoder_lstm2_output = decoder_lstm2(decoder_lstm1_output)
#     decoder_lstm3_output = decoder_lstm3(decoder_lstm2_output)

#     flatten = tf.keras.layers.Flatten()(decoder_lstm3_output)
#     model_output = tf.keras.layers.Dense(1)(flatten)
    
#     model = tf.keras.models.Model(model_input, model_output)
    
#     return model

In [ ]:
def seq2seq_model(input_shape):
    model_input = tf.keras.layers.Input(shape=input_shape)

    # for feature extracting
    conv1 = tf.keras.layers.Conv1D(64, 1, activation='swish')(model_input)
    pool1 = tf.keras.layers.MaxPool1D(pool_size=2, strides=1, padding='same')(conv1)
    bat01 = tf.keras.layers.BatchNormalization()(pool1)
    conv2 = tf.keras.layers.Conv1D(32, 1, activation='swish')(bat01)
    pool2 = tf.keras.layers.MaxPool1D(pool_size=2, strides=1, padding='same')(conv2)
    bat02 = tf.keras.layers.BatchNormalization()(pool2)

    # 인코더 - 디코더 선언
    encoder_lstm1 = tf.keras.layers.LSTM(16, return_sequences=True, activation='swish')
    encoder_lstm2 = tf.keras.layers.LSTM(32, return_sequences=True, activation='swish')
    encoder_lstm3 = tf.keras.layers.LSTM(64, return_state=True, return_sequences=True, activation='swish')

    decoder_lstm1 = tf.keras.layers.LSTM(64, return_sequences=True, activation='swish')
    decoder_lstm2 = tf.keras.layers.LSTM(32, return_sequences=True, activation='swish')
    decoder_lstm3 = tf.keras.layers.LSTM(16, return_sequences=True, activation='swish')

    # 인코더
    encoder_output_lstm1 = encoder_lstm1(bat02)
    encoder_output_lstm2 = encoder_lstm2(bat01)
    encoder_output_lstm4, state_h, state_c = encoder_lstm3(encoder_output_lstm2)

    #디코더
    decoder_lstm1_output = decoder_lstm1(encoder_output_lstm4, initial_state=[state_h, state_c])
    decoder_lstm2_output = decoder_lstm2(decoder_lstm1_output)
    decoder_lstm3_output = decoder_lstm3(decoder_lstm2_output)

    flatten = tf.keras.layers.Flatten()(decoder_lstm3_output)
    model_output = tf.keras.layers.Dense(1)(flatten)
    
    model = tf.keras.models.Model(model_input, model_output)
    
    return model

In [ ]:
## , Features
model_kier = seq2seq_model(input_shape=(24, int_len_col_input))
model_kier.summary()

In [ ]:
from keras.callbacks import EarlyStopping, ModelCheckpoint
from keras.models import Sequential, load_model

# 모델 컴파일
earlystopping = EarlyStopping(monitor='loss', patience=15)
checkpoint = ModelCheckpoint(moniter='loss',filepath = str(str_domain + '_' + str(int_interval) + '_S2S.h5'))
model_kier.compile(loss='mae', optimizer=tf.keras.optimizers.Adamax(learning_rate=3e-5,clipnorm=1.0), metrics=['mse'])
# 모델 요약 정보 출력
hist_kier = model_kier.fit(trainX_kier, trainY_kier, epochs=500, batch_size=64, callbacks=[earlystopping,checkpoint])

In [ ]:
pred_kier = np.reshape(model_kier.predict(testX_kier), (-1, 1))
real_kier = testY_kier

In [ ]:
com_Model.model_sk_metrics(real_kier, pred_kier)
# list_scores.append(tm_code)

tm_code = time.time() - tm_start
print("time : ", tm_code)

In [ ]:
print(str_domain, ' /// ', str_interval, ' /// ', str_col_tar)

plt.figure(figsize=(20,4))
plt.plot(real_kier, color='red', label='True')
plt.plot(pred_kier, color='blue', label='Pred')
plt.legend()
plt.show()

plt.figure(figsize=(20,4))
plt.plot(real_kier, color='red', label='True')
plt.plot(pred_kier, color='blue', label='Pred')
plt.xlim(0, 1000)
plt.legend()
plt.show()

plt.figure(figsize=(20,4))
plt.plot(pred_kier, color='blue', label='Pred')
plt.plot(real_kier, color='red', label='True')
plt.legend()
plt.show()

plt.figure(figsize=(20,4))
plt.plot(pred_kier, color='blue', label='Pred')
plt.plot(real_kier, color='red', label='True')
plt.xlim(0, 1000)
plt.legend()
plt.show()